In [1]:
#import the necessary packages for all analysis
import pandas as pd
import numpy as np 

In [2]:
#Import the data from Kaggle 
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/test.csv')
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/train.csv')

# Titanic Machine Learning - Random Forest

In [3]:
#now import the sklearn packages for splitting the sample and for running the Logit analysis 
#for splitting test/train and for k-folds cross validation 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV 

#to make transforming the data easier ColumnTransformer and Pipeline keep preprocessed data inside the folds 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#for encoding the data 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#to actually run the model 
#from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier

#to test how well the model performs 
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score 

In [4]:
#first create additional features
def add_titanic_features(df: pd.DataFrame) -> pd.DataFrame: 
    df = df.copy()

    #add log(fare) 
    df["log_Fare"] = np.log1p(df["Fare"])

    #family features (total size and if they're alone) 
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    #title from name 
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand = False).str.strip()
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss", "Ms":"Miss","Mme":"Mrs"
    })
    df["Title"] = df["Title"].where(df["Title"].isin(["Mr","Mrs","Miss","Master"]), "Other")

    #Cabin Features 
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")

    return df

In [5]:
#now add the additional covariates to the training data 
train_fe = add_titanic_features(train_df)

#define the feature list, then the X and y 
Feature_List = [
    "Pclass", "Sex", "Age",
    "log_Fare", "Embarked",
    "FamilySize", "IsAlone",
    "Title",
    "CabinKnown", "Deck",
]

X = train_fe[Feature_List]
y = train_fe["Survived"]

In [10]:
#now build the preprocessing 
numeric_features = ["Age","log_Fare","FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked",  "Title", "Deck"]
binary_features = ["IsAlone","CabinKnown"]
#numeric preprocessing
numeric_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy  = "median", add_indicator = True)),
                                         ]) # don't need scaling for RF

#categorical preprocessing 
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "Missing")),
                                             ("onehot", OneHotEncoder(handle_unknown = "ignore"))]) # don't need to drop for RF

#binary preprocessing 
binary_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent", add_indicator = True))])
#preprocess 
preprocess = ColumnTransformer(transformers = [("num", numeric_transformer, numeric_features),
                                              ("cat",categorical_transformer, categorical_features),
                                              ("bin", binary_transformer, binary_features)])


## Random Forest - Simple 80-20 Split

In [13]:
#split the sample
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 1234, stratify = y)

#define the random forest model 
rf_model = RandomForestClassifier(n_estimators = 500,
                                 random_state = 1234, 
                                 n_jobs = -1,
                                 max_features="sqrt",
                                 min_samples_leaf=5,
                                 min_samples_split=10,
                                 max_depth = None)

random_forest = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", rf_model)])

#fit the model 
random_forest.fit(X_train, y_train)

#Evaluate the model
probability_rf = random_forest.predict_proba(X_val)[:,1]
prediction_rf = (probability_rf > 0.5).astype(int)

#RF Evaluation 
rf_eval = {
    "log_loss": log_loss(y_val, probability_rf),
    "roc_auc": roc_auc_score(y_val, probability_rf),
    "accuracy": accuracy_score(y_val, prediction_rf)
}
print("Random Forest:" ,rf_eval)
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

Random Forest: {'log_loss': 0.43592176595597953, 'roc_auc': 0.8501976284584981, 'accuracy': 0.8268156424581006}
Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}


In [16]:
# With the given hyperparameters random forest does well, but we can tune our hyperparameters to get better performance. 
# What hyperparameters do i want to tune over?
# - Max Features: or how many features the RF is allowed to consider at each node (high = overfit)
# - Min Samples Leaf: how many observations i have to have in each terminal bucket (low = overfit) 
# - Min Samples Split: how many observations in each node i have to have to split (low = overfit) goes hand in hand with min samples leaf
# - Max Depth: how deep my tree can go on splits. 

# First set up the RF pipeline without any parameters set 
rf_model_tune = RandomForestClassifier(n_estimators = 500,
                                 random_state = 1234, 
                                 n_jobs = -1)
# set the parameter space we want to search over
param_grid = {
    "model__max_features":["sqrt", 0.5, None],
    "model__min_samples_leaf":[1,2,5,10],
    "model__min_samples_split":[2,5,10,20],
    "model__max_depth":[None,5,10,15]
}

#now set the inner cross validation 5-folds. 
inner_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 1234)

#search the parameter space 
rf_search = GridSearchCV(
    rf_model_tune,
    param_grid = param_grid, 
    scoring = "roc_auc",
    cv = inner_cv,
    n_jobs = -1
)

#find the best parameters 
rf_search.fit(X_train,y_train)
best_rf = rf_search.best_estimator_ 
print("Best Parameters:", rf_search.best_params_)

#Then evaluate the best model: 
#Evaluate the model
probability_rf_tune = best_rf.predict_proba(X_val)[:,1]
prediction_rf_tune = (probability_rf_tune > 0.5).astype(int)

#RF Evaluation 
rf_eval_tune = {
    "log_loss": log_loss(y_val, probability_rf_tune),
    "roc_auc": roc_auc_score(y_val, probability_rf_tune),
    "accuracy": accuracy_score(y_val, prediction_rf_tune)
}
print("Random Forest (simple):" ,rf_eval)
print("Random Forest (tuned):" ,rf_eval_tune)
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

Best Parameters: {'model__max_depth': 5, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}
Random Forest (simple): {'log_loss': 0.43592176595597953, 'roc_auc': 0.8501976284584981, 'accuracy': 0.8268156424581006}
Random Forest (tuned): {'log_loss': 0.472719480359347, 'roc_auc': 0.8177865612648221, 'accuracy': 0.8324022346368715}
Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}


# Random Forest with K-Fold Cross Validation

In [18]:
#A simple 80-20 holdout doesn't optimize parameters ideally so we need to do k-folds 
#now repeat for the Lasso (L1 Model)
#set up the outer k-fold and the inner k-fold. the outer is for model evaluation the inner is for c parameter choice 
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

#Set up the parameter grid same as before 
parameter_grid = {
    "model__max_features":["sqrt", 0.5, None],
    "model__min_samples_leaf":[1,2,5,10],
    "model__min_samples_split":[2,5,10,20],
    "model__max_depth":[None,5,10,15]
}

#define the lasso pipeline 
rf_tune_kfold = Pipeline(steps = [
    ("preprocess", preprocess),
    ("model", rf_model_tune)
])

rf_tune_search = GridSearchCV(
    rf_tune_kfold, 
    param_grid = parameter_grid,
    scoring = "roc_auc", 
    cv = inner_cv, 
    n_jobs = -1)

#now run the cross validation 
scoring = {
    'logloss': 'neg_log_loss',
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy'}

cv_results_RF = cross_validate(
    rf_tune_search, X, y, 
    cv = outer_cv, 
    scoring = scoring, #same scoring as above with the logit
    return_train_score = False 
)

#now calculate and print out the model quality 
print("Logit K-Fold Cross Validation Evaluation", "\n", "5-fold CV Log Loss: 0.4357 ± 0.0067","\n","5-fold CV ROC-AUC:  0.8695 ± 0.0182",
    "\n","5-fold CV Accuracy: 0.8261 ± 0.0276")
print("Nested RF Hypertune")
print("Log loss:", (-cv_results_RF["test_logloss"]).mean(), "±", (-cv_results_RF["test_logloss"]).std())
print("ROC-AUC: ", (cv_results_RF["test_roc_auc"]).mean(), "±", (cv_results_RF["test_roc_auc"]).std())
print("Accuracy:", (cv_results_RF["test_accuracy"]).mean(), "±", (cv_results_RF["test_accuracy"]).std())

Logit K-Fold Cross Validation Evaluation 
 5-fold CV Log Loss: 0.4357 ± 0.0067 
 5-fold CV ROC-AUC:  0.8695 ± 0.0182 
 5-fold CV Accuracy: 0.8261 ± 0.0276
Nested RF Hypertune
Log loss: 0.4168465727861882 ± 0.02177800290279482
ROC-AUC:  0.8789395752499063 ± 0.008756468416159765
Accuracy: 0.8350260498399347 ± 0.011422552900554295
